# Diffusion foundations: from a known corruption process to a learned denoiser

Diffusion training is unusually concrete. We choose a forward process that adds Gaussian noise, sample a random time step, and train a network to predict the noise that was added. The network never needs a human label for the clean sample beyond the sample itself.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
sns.set_theme(style='whitegrid',context='notebook'); torch.manual_seed(5); np.random.seed(5)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T=100; beta=torch.linspace(1e-4,.02,T); alpha=1-beta; alpha_bar=torch.cumprod(alpha,0)

## 1. The forward process

With `ε ~ N(0,I)`, the closed form is

`x_t = √ᾱ_t x_0 + √(1−ᾱ_t) ε`.

`ᾱ_t` is the cumulative signal retained after all earlier steps. The closed form lets us jump directly to any time step during training instead of simulating every previous corruption.

In [ ]:
x0=torch.linspace(-1,1,500); noise=torch.randn_like(x0); steps=[0,9,24,49,99]
fig,axes=plt.subplots(1,len(steps),figsize=(14,3),sharey=True)
for ax,t in zip(axes,steps):
    xt=alpha_bar[t].sqrt()*x0+(1-alpha_bar[t]).sqrt()*noise
    ax.scatter(x0,xt,s=4,alpha=.35); ax.set_title(f't={t+1}\nᾱ={alpha_bar[t]:.3f}'); ax.set_xlabel('x₀')
axes[0].set_ylabel('xₜ'); plt.tight_layout()

## 2. Learn the noise, not the clean point

The common epsilon objective is

`L = E_{x₀, t, ε} || ε − ε_θ(x_t,t) ||²`.

The time input matters because the same noisy value means different things early and late in the schedule. A sinusoidal embedding gives the network a smooth representation of time.

In [ ]:
class SinusoidalTime(nn.Module):
    def __init__(self,width):
        super().__init__(); self.width=width
    def forward(self,t):
        half=self.width//2; freqs=torch.exp(-torch.arange(half,device=t.device)*np.log(10000)/(half-1))
        angles=t.float().unsqueeze(1)*freqs.unsqueeze(0); return torch.cat([angles.sin(),angles.cos()],1)
class EpsilonModel(nn.Module):
    def __init__(self,width=64):
        super().__init__(); self.time=SinusoidalTime(width); self.net=nn.Sequential(nn.Linear(1+width,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,1))
    def forward(self,x,t): return self.net(torch.cat([x.unsqueeze(1),self.time(t/(T-1))],1)).squeeze(1)
model=EpsilonModel().to(device); opt=torch.optim.Adam(model.parameters(),lr=2e-3)
def clean_batch(n): return torch.where(torch.rand(n,device=device)>.5,torch.randn(n,device=device)*.25+1,torch.randn(n,device=device)*.25-1)
losses=[]
for step in range(1800):
    t=torch.randint(0,T,(256,),device=device); clean=clean_batch(256); eps=torch.randn_like(clean)
    noisy=alpha_bar[t].to(device).sqrt()*clean+(1-alpha_bar[t]).to(device).sqrt()*eps
    loss=nn.functional.mse_loss(model(noisy,t),eps); opt.zero_grad(); loss.backward(); opt.step()
    if step%50==0: losses.append((step,float(loss)))
print('final training loss:',float(loss)); plt.plot(*zip(*losses)); plt.xlabel('step'); plt.ylabel('epsilon MSE'); plt.title('Noise-prediction training'); plt.show()

## 3. The reverse update

Given a predicted noise, estimate the clean sample:

`x̂₀ = (x_t − √(1−ᾱ_t) ε_θ(x_t,t)) / √ᾱ_t`.

A DDPM reverse step uses a learned or fixed Gaussian posterior whose mean combines `x_t` and `x̂₀`, and adds noise except at the final step. The next lesson derives those coefficients and compares this stochastic sampler with DDIM.

### Limits

The scalar experiment is not an image generator. It isolates the schedule, time conditioning, and objective. Image diffusion adds a U-Net, spatial tensors, residual blocks, attention, and a much larger data and compute budget.